# RAG (Retrieval Augmented Generation) Implementation with Microsoft Fabric

## Overview
This notebook demonstrates how to build a **RAG (Retrieval Augmented Generation)** pipeline using **Microsoft Fabric**, **Azure OpenAI**, and **Azure AI Search**.

### What is RAG?
RAG is an AI pattern that enhances Large Language Model (LLM) responses by retrieving relevant information from your **own knowledge base** before generating an answer. Instead of relying solely on the model's training knowledge, RAG **grounds** responses in your specific data.

### Pipeline Architecture
| Step | What Happens |
|------|-------------|
| 1. Ingest | Load a diabetes FAQ CSV into a Spark DataFrame |
| 2. Chunk | Split long text into smaller pieces using SynapseML's `PageSplitter` |
| 3. Embed | Convert each chunk to a vector using Azure OpenAI `text-embedding-ada-002` |
| 4. Index | Store vectors in Azure AI Search for fast semantic retrieval |
| 5. Query | At runtime — embed the question → retrieve top chunks → pass as context to GPT-4o → get a grounded answer |

### Azure Services Used
- **Azure OpenAI** — `text-embedding-ada-002` for embeddings, `gpt-4o` for response generation
- **Azure AI Search** — Vector + keyword hybrid search index
- **Microsoft Fabric / SynapseML** — Distributed Spark processing and ML pipeline orchestration


### Step 1: Install SynapseML

**SynapseML** (formerly MMLSpark) is an open-source library that makes it easy to integrate Azure AI services with Apache Spark.

We use it in this notebook for three things:
- `PageSplitter` — splits long text documents into fixed-size, overlapping chunks
- `OpenAIEmbedding` — calls Azure OpenAI to generate embedding vectors at Spark scale (in parallel)
- `OpenAIChatCompletion` — calls GPT-4o to generate answers from a prompt

> **Note:** If this is your first time running the notebook, a kernel restart may be required after installation.


In [ ]:
# Install SynapseML — provides PageSplitter, OpenAIEmbedding, and OpenAIChatCompletion
# python-dotenv loads credentials from .env (same keys as embed.py)
# If running for the first time, a session restart may be needed after this completes
%pip install synapseml python-dotenv


### Step 2: Load the Knowledge Base (CSV Dataset)

We load the **Diabetes Treatment FAQ** CSV file from the Fabric Lakehouse `Files` section into a Spark DataFrame.

- The file is located at `Files/diabetes_treatment_faq.csv` in the attached Lakehouse (`GenAILH`)
- It contains two key columns:
  - **`Topic`** — the subject/category of the FAQ entry (e.g., "Blood Sugar Monitoring")
  - **`Description`** — the full-text answer/explanation that will be chunked and indexed

This DataFrame forms the **raw knowledge base** for our RAG system.


In [ ]:
# Load the diabetes FAQ CSV from the Fabric Lakehouse Files section
# 'header=true' tells Spark to use the first row as column names
# Expected columns: 'Topic' (category) and 'Description' (full-text answer)
df = spark.read.format("csv").option("header","true").load("Files/diabetes_treatment_faq.csv")
display(df)


StatementMeta(, 11cb16d4-023f-4298-a99d-cd4189a59722, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7bfc4ad2-b3f5-4858-aad2-03a5f260a749)

### Step 3: Configure Azure Service Credentials

Load keys and endpoints from a `.env` file (project root, parent directory, or `/lakehouse/default/Files/.env` in Fabric). See the repository `README.md` for the full list of variables.


In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

# Load .env from cwd, parent folder, or Fabric lakehouse Files
_candidate_paths = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
_lh = Path("/lakehouse/default/Files")
if _lh.exists():
    _candidate_paths.append(_lh / ".env")

_loaded = False
for _env_path in _candidate_paths:
    if _env_path.is_file():
        load_dotenv(_env_path)
        _loaded = True
        break
if not _loaded:
    load_dotenv()


def _resource_name_from_openai_base(base: str) -> str:
    if not base:
        return ""
    b = base.strip().rstrip("/")
    if "://" not in b:
        b = "https://" + b
    host = urlparse(b).netloc
    if ".openai.azure.com" in host:
        return host.split(".openai.azure.com")[0]
    return ""


# SynapseML chat completions (Azure OpenAI resource)
AZURE_OPENAI_KEY = os.environ.get("AZURE_OPENAI_KEY") or os.environ.get("OPENAI_API_KEY", "")
AZURE_OPENAI_RESOURCE_NAME = (
    os.environ.get("AZURE_OPENAI_RESOURCE_NAME", "").strip()
    or _resource_name_from_openai_base(os.environ.get("OPENAI_API_BASE", ""))
)

# Azure AI Search
AI_SEARCH_NAME = os.environ.get("AI_SEARCH_NAME", "")
AI_SEARCH_INDEX_NAME = os.environ.get("AI_SEARCH_INDEX_NAME", "")
AI_SEARCH_API_KEY = os.environ.get("AI_SEARCH_API_KEY", "")

# Embedding resource (same names as embed.py)
OPENAI_API_EMBED_KEY = os.environ.get("OPENAI_API_EMBED_KEY", "")
_embed_base = os.environ.get("OPENAI_API_EMBED_BASE", "").strip().rstrip("/")
OPENAI_API_EMBED_BASE = f"{_embed_base}/" if _embed_base else ""
OPENAI_EMBED_MODEL = os.environ.get("OPENAI_EMBED_MODEL", "text-embedding-ada-002")
OPENAI_API_EMBED_VERSION = os.environ.get(
    "OPENAI_API_EMBED_VERSION", os.environ.get("OPENAI_API_VERSION", "2024-02-01")
)

RAG_CHAT_DEPLOYMENT = os.environ.get("RAG_CHAT_DEPLOYMENT", "gpt-4o")

_missing = [
    name
    for name, ok in [
        ("OPENAI_API_KEY or AZURE_OPENAI_KEY", bool(AZURE_OPENAI_KEY)),
        ("AZURE_OPENAI_RESOURCE_NAME or OPENAI_API_BASE", bool(AZURE_OPENAI_RESOURCE_NAME)),
        ("AI_SEARCH_NAME", bool(AI_SEARCH_NAME)),
        ("AI_SEARCH_INDEX_NAME", bool(AI_SEARCH_INDEX_NAME)),
        ("AI_SEARCH_API_KEY", bool(AI_SEARCH_API_KEY)),
        ("OPENAI_API_EMBED_KEY", bool(OPENAI_API_EMBED_KEY)),
        ("OPENAI_API_EMBED_BASE", bool(OPENAI_API_EMBED_BASE)),
    ]
    if not ok
]
if _missing:
    raise ValueError("Missing required environment variables: " + ", ".join(_missing))


### Step 4: Split Text into Chunks

LLMs and embedding models have a **context window limit** — they cannot process very long texts at once. To handle this, we split the `Description` text into smaller, manageable pieces called **chunks**.

We use SynapseML's **`PageSplitter`** transformer:
- `setMaximumPageLength(4000)` — each chunk is at most 4000 characters
- `setMinimumPageLength(30)` — discard very short fragments (e.g., stray newlines)
- Output: a new `chunks` column containing an **array of strings** per row — one array per document

Smaller chunks improve embedding quality because the resulting vector captures the meaning of a focused piece of text rather than a long mixed paragraph.


In [ ]:
from synapse.ml.featurize.text import PageSplitter

# Configure the PageSplitter transformer
# - setInputCol: the column containing the long text to split ('Description')
# - setMaximumPageLength: max characters per chunk (stays within embedding model limits)
# - setMinimumPageLength: skip fragments shorter than this (avoids noise in the index)
# - setOutputCol: output column name — will contain an array of chunk strings per row
ps = (
    PageSplitter()
    .setInputCol("Description")
    .setMaximumPageLength(4000)
    .setMinimumPageLength(30)
    .setOutputCol("chunks")
)

# Apply the splitter — each row in splitted_df now has a 'chunks' array column
splitted_df = ps.transform(df)
display(splitted_df)


StatementMeta(, 11cb16d4-023f-4298-a99d-cd4189a59722, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3447cc2c-2734-426d-89f3-2757f40686cf)

### Step 5: Explode Chunks into Individual Rows

After splitting, each row has a `chunks` column containing an **array** of text strings. For embedding and indexing, we need **one chunk per row**.

We use Spark's **`posexplode`** function to flatten the array:
- `posexplode(chunks)` → produces two new columns: `chunk_index` (position in the array) and `chunk` (the text)
- We then create a `unique_id` column by combining `Topic` + `chunk_index`, which will serve as the document key in the Azure AI Search index (each document in the index must have a unique ID)

**Before:** 1 row → 1 array of N chunks  
**After:** N rows → 1 chunk per row


In [ ]:
from pyspark.sql.functions import posexplode, col, concat

# posexplode flattens the 'chunks' array into separate rows
# It produces two new columns:
#   - 'chunk_index': the position of the chunk within the original document (0, 1, 2, ...)
#   - 'chunk': the actual text content of that chunk
exploded_df = splitted_df.select("Topic", posexplode(col("chunks")).alias("chunk_index", "chunk"))

# Create a unique document ID for each chunk by combining Topic + chunk_index
# This ID will be used as the key field in the Azure AI Search index
# Example: "Blood Sugar Monitoring0", "Blood Sugar Monitoring1", etc.
exploded_df = exploded_df.withColumn("unique_id", concat(exploded_df.Topic, exploded_df.chunk_index))

display(exploded_df)


StatementMeta(, c6652cd0-e9e6-4654-8b28-942100a17313, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fd7bd150-0d90-467b-95be-43e6aa8376d0)

StatementMeta(, 11cb16d4-023f-4298-a99d-cd4189a59722, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 40e40482-cce2-45df-b15b-e3f833d21d2e)

### Step 6: Generate Vector Embeddings for Each Chunk

An **embedding** is a numerical vector representation of text that captures its **semantic meaning**. Chunks with similar meaning will have similar vectors, enabling semantic (meaning-based) search rather than just keyword matching.

We use Azure OpenAI's **`text-embedding-ada-002`** model which outputs a **1536-dimensional vector** per input.

SynapseML's `OpenAIEmbedding` transformer runs this in parallel across all Spark partitions, making it efficient even for large datasets.

> **Note:** The embedding model uses a **separate Azure OpenAI resource** (`mayan-mn36m84q-eastus2`) with its own key — this is why we define separate embedding credentials below.


Embedding variables (`OPENAI_API_EMBED_*`, `OPENAI_EMBED_MODEL`) are loaded from `.env` in the **configuration** cell above (same keys as `embed.py`).


In [ ]:
from synapse.ml.services import OpenAIEmbedding

# OpenAIEmbedding is a Spark ML transformer — it applies the embedding model
# to every row in the DataFrame in parallel across Spark partitions
embedding = (
    OpenAIEmbedding()
    .setDeploymentName(OPENAI_EMBED_MODEL)  # Azure OpenAI deployment name
    .setTextCol("chunk")                    # Input: text column to embed
    .setErrorCol("error")                   # Output: captures any API errors per row
    .setOutputCol("embeddings")             # Output: the resulting 1536-dim float vector
    .setApiVersion(OPENAI_API_EMBED_VERSION)
    .setSubscriptionKey(f"{OPENAI_API_EMBED_KEY}")
    .setEndpoint(f"{OPENAI_API_EMBED_BASE}")
)

# Apply the transformer — each row now has an 'embeddings' column with a dense vector
df_embeddings = embedding.transform(exploded_df)

display(df_embeddings)


### Step 7: Create a Vector Search Index in Azure AI Search

Before uploading any data, we define the **index schema** in Azure AI Search. This is like creating a table structure in a database — it tells Azure AI Search what fields to expect and how to handle them.

The index has three fields:
| Field | Type | Purpose |
|-------|------|---------|
| `id` | `Edm.String` (key) | Unique identifier for each chunk document |
| `content` | `Edm.String` | The raw text of the chunk (used for keyword search) |
| `contentVector` | `Collection(Edm.Single)` | The 1536-dimensional embedding vector (used for semantic search) |

The vector search uses the **HNSW** (Hierarchical Navigable Small World) algorithm with **cosine** similarity — an industry-standard approach for fast approximate nearest-neighbour search.

> If the index already exists, a `204 No Content` response means it was updated successfully.


In [ ]:
import requests
import json

# text-embedding-ada-002 produces 1536-dimensional vectors
EMBEDDING_LENGTH = 1536

# Build the REST API URL for creating (or updating) the index
# Using PUT method — creates the index if it doesn't exist, or updates it if it does
url = f"https://{AI_SEARCH_NAME}.search.windows.net/indexes/{AI_SEARCH_INDEX_NAME}?api-version=2023-11-01"

payload = json.dumps(
    {
        "name": AI_SEARCH_INDEX_NAME,
        "fields": [
            # Key field — must be a string; used to uniquely identify each document
            {
                "name": "id",
                "type": "Edm.String",
                "key": True,
                "filterable": True,
            },
            # Text content of the chunk — enables keyword-based (BM25) full-text search
            {
                "name": "content",
                "type": "Edm.String",
                "searchable": True,
                "retrievable": True,
            },
            # Vector field — stores the 1536-dim embedding; enables semantic similarity search
            {
                "name": "contentVector",
                "type": "Collection(Edm.Single)",
                "searchable": True,
                "retrievable": True,
                "dimensions": EMBEDDING_LENGTH,
                "vectorSearchProfile": "vectorConfig",  # matches the profile defined below
            },
        ],
        # Define the vector search algorithm (HNSW = fast approximate nearest-neighbour)
        "vectorSearch": {
            "algorithms": [{"name": "hnswConfig", "kind": "hnsw", "hnswParameters": {"metric": "cosine"}}],
            "profiles": [{"name": "vectorConfig", "algorithm": "hnswConfig"}],
        },
    }
)

headers = {"Content-Type": "application/json", "api-key": AI_SEARCH_API_KEY}

response = requests.request("PUT", url, headers=headers, data=payload)

# 201 = index created; 204 = index already existed and was updated
if response.status_code == 201:
    print("Index created!")
elif response.status_code == 204:
    print("Index updated!")
else:
    print(f"HTTP request failed with status code {response.status_code}")
    print(f"HTTP response body: {response.text}")


StatementMeta(, 11cb16d4-023f-4298-a99d-cd4189a59722, 24, Finished, Available, Finished, False)

Index updated!


### Step 8: Upload Chunks and Embeddings to Azure AI Search

Now we populate the index with our data. Each document uploaded to Azure AI Search includes:
- `id` — sanitised unique ID (special characters replaced with `_`)
- `content` — the raw text chunk
- `contentVector` — the embedding vector as a list of floats

Key implementation details:
- **Batch size of 1000** — Azure AI Search's REST API accepts at most 1000 documents per request
- **`mapPartitions`** — uploads are run in parallel across Spark partitions for efficiency
- **`make_safe_id`** — strips characters not allowed in Azure AI Search document keys (only `a-z`, `A-Z`, `0-9`, `-`, `_` are valid)

After this step, the knowledge base is fully indexed and ready to be queried.


In [ ]:
import re

from pyspark.sql.functions import monotonically_increasing_id


def insert_into_index(documents):
    """Sends a batch of documents to Azure AI Search using the index documents REST API."""
    url = f"https://{AI_SEARCH_NAME}.search.windows.net/indexes/{AI_SEARCH_INDEX_NAME}/docs/index?api-version=2023-11-01"

    payload = json.dumps({"value": documents})
    headers = {
        "Content-Type": "application/json",
        "api-key": AI_SEARCH_API_KEY,
    }

    response = requests.request("POST", url, headers=headers, data=payload)

    if response.status_code == 200 or response.status_code == 201:
        return "Success"
    else:
        return f"Failure: {response.text}"


def make_safe_id(row_id: str):
    """Sanitises a string for use as an Azure AI Search document key.
    Only alphanumeric characters, hyphens, and underscores are allowed.
    """
    return re.sub("[^0-9a-zA-Z_-]", "_", row_id)


def upload_rows(rows):
    """Generator function run on each Spark partition to upload documents in batches.
    Azure AI Search accepts at most 1000 documents per API call — we batch accordingly.
    Yields [start_index, end_index, status] tuples for progress tracking.
    """
    BATCH_SIZE = 1000
    rows = list(rows)
    for i in range(0, len(rows), BATCH_SIZE):
        row_batch = rows[i : i + BATCH_SIZE]
        documents = []
        for row in rows:
            documents.append(
                {
                    "id": make_safe_id(row["unique_id"]),       # Sanitised unique key
                    "content": row["chunk"],                     # Raw text chunk
                    "contentVector": row["embeddings"].tolist(), # Embedding as plain list
                    "@search.action": "upload",                  # 'upload' = upsert
                },
            )
        status = insert_into_index(documents)
        yield [row_batch[0]["row_index"], row_batch[-1]["row_index"], status]


# Add a row_index column so we can track which rows were uploaded in each batch
df_embeddings = df_embeddings.withColumn("row_index", monotonically_increasing_id())

# Run upload_rows on each Spark partition in parallel using mapPartitions
res = df_embeddings.rdd.mapPartitions(upload_rows)
display(res.toDF(["start_index", "end_index", "insertion_status"]))


StatementMeta(, 11cb16d4-023f-4298-a99d-cd4189a59722, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f0f1409d-8cc0-4bf3-a907-ee7ff4337964)

### Step 9: Embed the User's Question

At **query time**, we need to convert the user's question into the same vector space as the indexed chunks so we can measure semantic similarity.

This function uses the **same `text-embedding-ada-002` model** that was used to embed the chunks. This ensures both the question and the documents exist in the same semantic vector space, making similarity comparison meaningful.

If the embedding API call fails (e.g., wrong key or deployment), the function raises a clear error with diagnostic information instead of silently returning wrong results.


In [ ]:
def gen_question_embedding(user_question):
    """Generates a vector embedding for the user's question using SynapseML + Azure OpenAI.
    Uses the same text-embedding-ada-002 model as the indexed chunks so vectors are comparable.
    Returns the embedding as a plain Python list of floats.
    """
    from synapse.ml.services import OpenAIEmbedding

    # Wrap the question string in a single-row Spark DataFrame (required by SynapseML transformer)
    df_ques = spark.createDataFrame([(user_question,)], ["questions"])

    embedding = (
        OpenAIEmbedding()
        .setDeploymentName(OPENAI_EMBED_MODEL)
        .setTextCol("questions")        # Column containing the question text
        .setErrorCol("errorQ")          # Capture API errors here
        .setOutputCol("embeddings")     # Output column for the vector
        .setApiVersion(OPENAI_API_EMBED_VERSION)
        .setSubscriptionKey(f"{OPENAI_API_EMBED_KEY}")
        .setEndpoint(f"{OPENAI_API_EMBED_BASE}")
    )

    df_ques_embeddings = embedding.transform(df_ques)
    row = df_ques_embeddings.collect()[0]
    vec = row["embeddings"]

    # If the API call failed, 'embeddings' will be None — raise a clear error with diagnostics
    if vec is None:
        err = row["errorQ"]
        raise RuntimeError(
            "OpenAIEmbedding returned no vector (embeddings is null). "
            "Verify OPENAI_API_EMBED_KEY, OPENAI_API_EMBED_BASE, OPENAI_EMBED_MODEL, and OPENAI_API_EMBED_VERSION. "
            "SynapseML error column: "
            f"{err!r}"
        )

    # Convert the Spark DenseVector to a plain Python list for use in the search API payload
    return vec.tolist()


### Step 10: Retrieve the Most Relevant Chunks (Hybrid Search)

This function queries Azure AI Search using **hybrid search** — combining two complementary approaches:

- **Keyword search** (`"search": question`) — finds documents with matching words using BM25 scoring
- **Vector search** (`"vectorQueries"`) — finds semantically similar documents by comparing embedding vectors

The combination improves recall: keyword search catches exact matches while vector search catches paraphrases and synonyms. Together they return the `top-K` most relevant chunks from the knowledge base.


In [ ]:
import json 
import requests

def retrieve_top_chunks(k, question, question_embedding):
    """Queries Azure AI Search using hybrid search to find the top-K most relevant chunks.
    
    Hybrid search combines:
    - Keyword search ('search' field): finds exact/near-exact word matches using BM25
    - Vector search ('vectorQueries'): finds semantically similar chunks using cosine similarity
    
    Args:
        k: number of results to return
        question: the raw text question (used for keyword search)
        question_embedding: the vector embedding of the question (used for vector search)
    """
    url = f"https://{AI_SEARCH_NAME}.search.windows.net/indexes/{AI_SEARCH_INDEX_NAME}/docs/search?api-version=2023-11-01"

    payload = json.dumps({
        "search": question,         # Keyword search query
        "top": k,                   # Total number of results to return
        "vectorQueries": [
            {
                "vector": question_embedding,   # The question's embedding vector
                "k": k,                         # Number of nearest vectors to retrieve
                "fields": "contentVector",      # The index field storing chunk embeddings
                "kind": "vector"
            }
        ]
    })

    headers = {
        "Content-Type": "application/json",
        "api-key": AI_SEARCH_API_KEY,
    }

    response = requests.request("POST", url, headers=headers, data=payload)

    # Parse and return the full JSON response (contains 'value' list of matching documents)
    output = json.loads(response.text)
    return output


StatementMeta(, 11cb16d4-023f-4298-a99d-cd4189a59722, 28, Finished, Available, Finished, False)

### Step 11: Build Context from Retrieved Chunks

This function orchestrates the **retrieval** half of the RAG pipeline:

1. Generates a vector embedding for the user's question
2. Calls Azure AI Search to retrieve the **top 5** most relevant chunks (hybrid search)
3. Extracts the `content` text from each search result and returns them as a list

These chunks will be injected into the GPT-4o prompt as **context** — this is what makes the response grounded in your data rather than the model's general training knowledge.

You can increase `retrieved_k` to retrieve more chunks for broader coverage, or decrease it to keep the prompt focused.


In [ ]:
def get_context(user_question, retrieved_k=5):
    """Retrieves the most relevant text chunks for the user's question.
    
    Pipeline:
    1. Embed the question using text-embedding-ada-002
    2. Query Azure AI Search (hybrid search) to find the top-K matching chunks
    3. Extract and return the 'content' text from each matching document
    
    Args:
        user_question: the raw question string from the user
        retrieved_k: how many relevant chunks to retrieve (default 5)
    Returns:
        list of strings — each string is a relevant text chunk from the knowledge base
    """
    # Step 1: Convert the question to a vector embedding
    question_embedding = gen_question_embedding(user_question)

    # Step 2: Run hybrid search against the Azure AI Search index
    output = retrieve_top_chunks(retrieved_k, user_question, question_embedding)

    # Step 3: Extract just the 'content' text field from each returned document
    # 'output["value"]' is the list of matching documents returned by Azure AI Search
    context = [chunk["content"] for chunk in output["value"]]

    return context


StatementMeta(, 11cb16d4-023f-4298-a99d-cd4189a59722, 29, Finished, Available, Finished, False)

### Step 12: Generate a Response Using GPT-4o

This function is the **generation** half of the RAG pipeline — it ties everything together:

1. **Retrieves context** — calls `get_context()` to fetch the most relevant chunks for the question
2. **Builds a system prompt** — injects the retrieved chunks as context into a structured prompt
3. **Calls GPT-4o** — uses SynapseML's `OpenAIChatCompletion` to send the conversation (system prompt + user question) to the model
4. **Extracts the answer** — pulls the response text from the nested `chat_completions.choices.message.content` field

The system prompt instructs GPT-4o to **only answer based on the provided context** and say "I don't know" if the answer isn't present — this prevents hallucination and keeps responses grounded in your data.


In [ ]:
from pyspark.sql import Row
from synapse.ml.services.openai import OpenAIChatCompletion


def make_message(role, content):
    """Creates a Spark Row representing a single chat message.
    'role' is typically 'system' (instructions) or 'user' (the question).
    """
    return Row(role=role, content=content, name=role)


def get_response(user_question):
    """Full RAG pipeline: retrieves context then generates a grounded answer via GPT-4o.

    Steps:
    1. Retrieve relevant chunks from Azure AI Search
    2. Build a system prompt that injects the chunks as context
    3. Call GPT-4o via SynapseML's OpenAIChatCompletion
    4. Extract and return the model's reply text
    """
    # Step 1: Get the top relevant text chunks for this question
    context = get_context(user_question)

    # Step 2: Build the system prompt — instruct the model to answer ONLY from the context
    # If the answer is not in the chunks, the model should say "I don't know"
    prompt = f"""
    context: {context}
    Answer the question based on the context above.
    If the information to answer the question is not present in the given context then reply "I don't know".
    """

    # Step 3: Build a single-row Spark DataFrame with the conversation messages
    # SynapseML expects messages as an array of Rows with fields: role, content, name
    chat_df = spark.createDataFrame(
        [
            (
                [
                    make_message("system", prompt),         # System message = instructions + context
                    make_message("user", user_question),    # User message = the actual question
                ],
            ),
        ]
    ).toDF("messages")

    # Step 4: Configure the GPT-4o chat completion transformer
    chat_completion = (
        OpenAIChatCompletion()
        .setDeploymentName(RAG_CHAT_DEPLOYMENT)  # Must match your Azure OpenAI deployment name (.env)
        .setMessagesCol("messages")             # Column containing the conversation messages
        .setErrorCol("error")                   # Column to capture API errors
        .setOutputCol("chat_completions")       # Column for the full API response
        .setSubscriptionKey(f"{AZURE_OPENAI_KEY}")
        .setEndpoint(f"https://{AZURE_OPENAI_RESOURCE_NAME}.openai.azure.com/")
    )

    # Step 5: Run the transformer — result_df contains the nested response structure
    # We select only the 'content' field from the nested path: chat_completions → choices → message → content
    result_df = chat_completion.transform(chat_df).select("chat_completions.choices.message.content")

    # Step 6: Extract the response text
    # 'content' is a list (one entry per choice); we guard against None in case of API errors
    result = []
    for row in result_df.collect():
        contents = row["content"]
        if contents:
            content_string = " ".join(str(c) for c in contents if c is not None)
            result.append(content_string)

    # Join all choices into a single response string
    result = " ".join(result)

    return result


### Step 13: Test the RAG System End-to-End

Now we run a sample question through the **complete RAG pipeline**:

```
User question → embed question → search index → retrieve top chunks → build prompt → GPT-4o → print answer
```

You can change `user_question` to test with any query about diabetes treatment. If the answer exists in the FAQ dataset, GPT-4o will return a grounded response. If it doesn't, it will respond with "I don't know".


In [ ]:
# Change this question to test different queries against the diabetes FAQ knowledge base
user_question = "how often should i check my blood sugar levels?"

# Run the full RAG pipeline: embed → search → retrieve → prompt → GPT-4o → answer
response = get_response(user_question)
print(response)


StatementMeta(, 11cb16d4-023f-4298-a99d-cd4189a59722, 38, Finished, Available, Finished, False)

The frequency of checking your blood sugar levels depends on factors like the type of diabetes you have and your treatment plan:

- **Type 1 Diabetes or Insulin Use**: You may need to test multiple times per day, including before and after meals or exercise.
- **Type 2 Diabetes**: The frequency varies based on your treatment (e.g., medication or lifestyle management) and how well your blood sugar is controlled.

It’s best to consult your healthcare provider for personalized recommendations that ensure your blood sugar levels remain within the target range to prevent complications.
